# DATALUS Generative Studio: Simulação de Cenários e GenAI Aplicada

Este notebook demonstra o **DATALUS Generative Studio**, com foco em tarefas de IA Generativa aplicada usando um modelo de difusão pré-treinado.
Exploramos como Cientistas de Dados e Gestores de Políticas Públicas podem aproveitar dados sintéticos para simulação de cenários,
balanceamento de classes minoritárias, inpainting tabular e aumento de dados avançado.

## 1. Configuração do Ambiente e Verificação de Artefatos
Detectando o ambiente e verificando os pré-requisitos. Se os artefatos de treinamento estiverem ausentes, executamos um passe de treinamento rápido.

In [ ]:
import os
import sys
from pathlib import Path
import polars as pl
import numpy as np

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_studio'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_studio'
    return './datalus_studio'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

# Instalar dependências se estiver no Colab/Kaggle
if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    !git clone https://github.com/emanuellcs/datalus.git || true
    %cd datalus
    !python -m pip install -e '.[dev]' pysus polars matplotlib seaborn
    sys.path.append(os.getcwd())

checkpoint_path = Path(f'{WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt')
if not checkpoint_path.exists():
    print('Training artifacts not found. Running lightning training pass...')
    from pysus.api import sih
    df = pl.from_pandas(sih(state='SP', year=2024, month=[1]))
    df = df.with_columns(pl.col('MORTE').cast(pl.Int64).fill_null(0)).head(5000)
    df.write_parquet(f'{WORKING_DIR}/raw_sample.parquet')
    
    !datalus ingest {WORKING_DIR}/raw_sample.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE
    !datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 1 --batch-size 256
else:
    print('Verified: Training artifacts are ready.')


## 2. Geração Ab-Initio e Análise de Distribuição (KDE)
Gerando um grande conjunto de dados sintéticos e comparando sua distribuição estatística com os dados reais usando estimativa de densidade de kernel (KDE).

In [ ]:
!datalus sample {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet --n-records 10000 --cfg-scale 1.0

import matplotlib.pyplot as plt
import seaborn as sns

real = pl.read_parquet(f'{WORKING_DIR}/processed.parquet')
synth = pl.read_parquet(f'{WORKING_DIR}/synthetic.parquet')

plt.figure(figsize=(12, 6))
sns.kdeplot(data=real['IDADE'], label='Real Data', fill=True, alpha=0.3)
sns.kdeplot(data=synth['IDADE'], label='Synthetic Data', fill=True, alpha=0.3)
plt.title('Kernel Density Estimation: Real vs Synthetic (Age / IDADE)')
plt.xlabel('Age (Years)')
plt.ylabel('Density')
plt.legend()
plt.show()

## 3. Aumento de Dados Avançado (Augmentation)
Demonstrando como o aumento sintético evita o overfitting em tarefas subsequentes de machine learning.

In [ ]:
seed = synth.sample(n=1000)
seed.write_parquet(f'{WORKING_DIR}/seed_augment.parquet')

!datalus augment {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/seed_augment.parquet {WORKING_DIR}/augmented.parquet --n-records 1000

## 4. Justiça Algorítmica: Balanceamento de Classes Minoritárias
Lidando com o desequilíbrio de classes (ex: doenças raras ou demografias específicas) através de sobreamostragem sem duplicação de linhas.

In [ ]:
!datalus balance {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/balanced.parquet MORTE '{"1": 5000}'

balanced = pl.read_parquet(f'{WORKING_DIR}/balanced.parquet')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
real['MORTE'].value_counts().to_pandas().plot(kind='bar', x='MORTE', y='count', ax=ax1, color='salmon')
ax1.set_title('Original Class Imbalance (Real)')
balanced['MORTE'].value_counts().to_pandas().plot(kind='bar', x='MORTE', y='count', ax=ax2, color='skyblue')
ax2.set_title('Balanced Class Distribution (Synthetic)')
plt.show()

## 5. Inpainting Tabular (Imputação de Dados Faltantes)
Usando inferência no estilo RePaint para preencher condicionalmente valores nulos enquanto preserva os campos observados.

In [ ]:
incomplete = real.head(100).with_columns(
    pl.when(pl.Series(np.random.rand(100) > 0.5)).then(None).otherwise(pl.col('IDADE')).alias('IDADE')
)
incomplete.write_parquet(f'{WORKING_DIR}/incomplete.parquet')

!datalus inpaint {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/incomplete.parquet {WORKING_DIR}/inpainted.parquet

## 6. Intervenções Contrafactuais e Simulação de Políticas
Simulando cenários "E se" ao modificar características do paciente e permitindo que o modelo regenere o registro clínico de forma coerente.

In [ ]:
!datalus counterfactual {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/counterfactual.parquet '{"SEXO": "1"}'